# K-Nearest Neighbors: Classification & Regression

This project applies K-Nearest Neighbors (KNN) across several datasets and problem types: basic classification with hyperparameter tuning, the effects of normalization and distance weighting, KNN regression, a discussion of an ethically inappropriate input feature in a classic dataset, and a custom mixed-type distance metric for data with both nominal and continuous features.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
import numpy as np
import pandas as pd
from scipy.io import arff
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score

## Basic KNN Classification

KNN classification (`KNeighborsClassifier`, default parameters) on the Glass Identification dataset, evaluated across random splits, train/test split ratios, predicted-class probabilities, and different Minkowski distance exponents (p-values).

In [ ]:
# Learn the glass data
glass_attributes = ["id",
                    "RI",
                    "Na",
                    "Mg",
                    "Al",
                    "Si",
                    "K",
                    "Ca",
                    "Ba",
                    "Fe",
                    "glass_type"
                    ]


# read the csv (using pandas)
glass_df = pd.read_csv("data/glass-identification/glass.data", header=None, names=glass_attributes)


#EDA
print("First 5 rows of glass data set:")
display(glass_df.head())

print("\nGlass Type Distribution:")
display(glass_df["glass_type"].value_counts().sort_index())

print(f"Data shape: {glass_df.shape}")

#split into x and y
X = glass_df.drop(["id", "glass_type"], axis = 1)
y = glass_df["glass_type"]


#knn model generation, training
def knn_split(X, y, test_size = 0.2, p = 2, k = 5, random_state = 0):
    """Split data, initialize and fit KNN using k=5.
    Returns KNN model, splits, and accuracies."""
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size = test_size, stratify=y, random_state=random_state
    )

    knn_model = KNeighborsClassifier(n_neighbors = k, p = p)
    knn_model.fit(X_train, y_train)
    train_acc = knn_model.score(X_train, y_train)
    test_acc = knn_model.score(X_test, y_test)

    return knn_model, X_train, X_test, y_train, y_test, train_acc, test_acc

#Run with Random States
split = 0.2
p = 2
k = 5
states = np.concatenate([np.array([0]), np.random.randint(0, 101, size=4)])
random_state_models = {}
random_state_results = []

for state in states:
    model, X_train, X_test, y_train, y_test, train_acc, test_acc = knn_split(
        X, y, split, p, k, state
    )
    #save model for each random state tested
    random_state_models[state] = (model, X_train, X_test, y_train, y_test)
    #save accuracies for each model
    random_state_results.append({"state": state, "train_acc": train_acc, "test_acc": test_acc})

random_state_result_df = pd.DataFrame(random_state_results).sort_values(by = "state")
random_state_result_df.set_index("state", inplace = True)
print("\nAccuracies using random states with p = 2:")
display(random_state_result_df)
print("Average train accuracy:", random_state_result_df["train_acc"].mean())
print("Average test accuracy:", random_state_result_df["test_acc"].mean())

#Run with Random Splits
p = 2
k = 5
splits = [0.15, 0.20, 0.30, 0.50, 0.80]
random_split_models = {}
random_split_results = []

for sp in splits:
    model, X_train, X_test, y_train, y_test, train_acc, test_acc = knn_split(
        X, y, sp, p, k
    )
    #save model for each random state tested
    random_split_models[sp] = (model, X_train, X_test, y_train, y_test)
    #save accuracies for each model
    random_split_results.append({"split": sp, "train_acc": train_acc, "test_acc": test_acc})

random_sp_result_df = pd.DataFrame(random_split_results)
random_sp_result_df.set_index("split", inplace = True)
print(f"\nAccuracies using random splits with p = 2 and random state = {states[0]}:")
display(random_sp_result_df)
print("Average train accuracy:", random_sp_result_df["train_acc"].mean())
print("Average test accuracy:", random_sp_result_df["test_acc"].mean())

#Predict Proba using one of the randomly generated states
p2_model, _, X_test, _, y_test = random_state_models[states[0]]
predictions = p2_model.predict(X_test.head())
probabilities = p2_model.predict_proba(X_test.head())

predict_proba_df = pd.DataFrame(
    probabilities, columns=[f"class_{c}" for c in p2_model.classes_]
)
predict_proba_df.insert(0, "predicted_label", predictions)
predict_proba_df.insert(0, "actual_label", y_test.head().values)
print("\nPredicted_proba:")
display(predict_proba_df)


#Try with different p values
split = 0.2
p_values = [1, 2, 3]
p_value_models = {}
p_value_results = []

for p in p_values:
    model, X_train, X_test, y_train, y_test, train_acc, test_acc = knn_split(
        X, y, split, p, k
    )
    #save model for each random state tested
    p_value_models[p] = (model, X_train, X_test, y_train, y_test)
    #save accuracies for each model
    p_value_results.append({"p": p, "train_acc": train_acc, "test_acc": test_acc})


p_value_result_df = pd.DataFrame(p_value_results)
p_value_result_df.set_index("p", inplace = True)
print("\nAccuries using a single 80/20 split per p:")
display(p_value_result_df)

**Results**

The initial exploratory analysis showed clean, uniform data with no missing values — a good candidate for classification without much preprocessing.

**Random samples:** Across 5 different random seeds, train and test accuracy stayed relatively close for most runs (with one outlier at seed 94, where test accuracy was notably higher than expected). The average gap between train and test accuracy was about 0.1, with train consistently higher — a mild sign of overfitting, though the overall consistency across seeds suggests the dataset is reasonably well balanced.

**Split ratios:** Testing splits of [0.15, 0.20, 0.30, 0.50, 0.80] with a fixed random seed showed train and test accuracy were nearly identical for 0.15, 0.20, 0.30, and 0.80 — a genuinely surprising result, since a much larger test proportion (0.80) would be expected to shift accuracy more than it did. The 0.50 split was the outlier, with accuracy about 0.1 lower than the others; the cause isn't clear from this data alone and would be worth investigating with different random states.

**Predicted probabilities:** `predict_proba` behaved as expected, correctly predicting the class with the highest calculated probability for each test point.

**p-values (Minkowski exponent):** p = 1 had the lowest test accuracy but the highest train accuracy. p = 2 and p = 3 had identical train accuracy, but p = 2 had the highest test accuracy. This is consistent with p = 2 (Euclidean distance) being scikit-learn's default — it appears to offer the best balance of accuracy and generalization for this dataset.

## Normalization and Distance Weighting (Magic Telescope Dataset)

KNN classification on the Magic Telescope dataset, comparing a baseline model (k=3, no normalization, no distance weighting) against normalized inputs and, separately, distance-weighted voting.

In [ ]:
# Learn magic telescope data
# Create Data Frames from the data files
train_data, train_meta = arff.loadarff("data/magic-telescope/magic_telescope_train.arff")
test_data, test_mets = arff.loadarff("data/magic-telescope/magic_telescope_test.arff")

tr_df = pd.DataFrame(train_data)
te_df= pd.DataFrame(test_data)

# Concatonate data sets in order to split into desired 80/20 later
magic_telescope_df = pd.concat([tr_df, te_df], ignore_index = True)

#EDA
print("First 5 rows of magic telescope data set:")
display(magic_telescope_df.head())

print("\nMagic Telescope Class Distribution:")
display(magic_telescope_df["class"].value_counts())

print(f"Data shape: {magic_telescope_df.shape}")

#split into x and y
X = magic_telescope_df.drop(["class"], axis = 1)

encoder = LabelEncoder()
y = encoder.fit_transform(magic_telescope_df["class"])

print("\nRunning KNNeighbors with k = 3, NO weighting, and NO normalization:")

model, X_train, X_test, y_train, y_test, train_acc, test_acc = knn_split(
        X, y, 0.2, 2, 3, random_state=0)

print(f"\n\tAverage train accuracy: {train_acc}")
print(f"\tAverage test accuracy: {test_acc}")

**Results**

Baseline (k=3, no normalization, no weighting): average train accuracy just under 0.90, average test accuracy around 0.80 — a solid starting point with clear room for improvement.

Note: the provided train/test files were concatenated and re-split, since both files include class labels and there was no explicit constraint against combining them. Re-running with just the original training set produced nearly identical accuracy, suggesting the provided train/test split was already well balanced.

### Adding Normalization

In [ ]:
# Train/Predict with normalization
scaler = StandardScaler()

# Normalize
X_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Initialize, fit, score a new model based on scaled data
knn_model = KNeighborsClassifier(n_neighbors = 3)
knn_model.fit(X_scaled, y_train)

train_acc = knn_model.score(X_scaled, y_train)
test_acc = knn_model.score(X_test_scaled, y_test)

print("\nRunning KNNeighbors with k = 3, NO weighting, and normalization:")
print(f"\n\tAverage train accuracy: {train_acc}")
print(f"\tAverage test accuracy: {test_acc}")

**Results**

Normalizing input features (via `StandardScaler`) increased both train and test accuracy by roughly 0.02–0.03 over the baseline — a modest but real improvement. The gap between train and test accuracy also narrowed slightly, suggesting normalization helped reduce a small amount of overfitting as well as bias from features on different scales.

### Adding Distance Weighting

In [ ]:
# Initialize, fit, score a new model based on previously scaled data, created in a function to use in the next part

knn_model = KNeighborsClassifier(n_neighbors = 3, weights = "distance")
knn_model.fit(X_scaled, y_train)

train_acc = knn_model.score(X_scaled, y_train)
test_acc = knn_model.score(X_test_scaled, y_test)

print("\nRunning KNNeighbors with k = 3, NO weighting, and normalization:")
print(f"\n\tAverage train accuracy: {train_acc}")
print(f"\tAverage test accuracy: {test_acc}")

**Results**

With distance weighting added on top of normalization, train accuracy reached 1.0 — the model perfectly fits its own training data. Test accuracy stayed close to the normalization-only result, but still improved over the original baseline. The near-0.2 gap between train and test accuracy here is a stronger overfitting signal than in the previous runs, though the net result is still an improvement in test performance.

### Varying k

In [ ]:
neighbors = [k for k in range(1, 16)]
test_accuracies = []
# Get test accuracies for each of the k values
for k in neighbors:
    knn_model = KNeighborsClassifier(n_neighbors = k, weights = "distance")
    knn_model.fit(X_scaled, y_train)
    test_acc = knn_model.score(X_test_scaled, y_test)
    test_accuracies.append(test_acc)

#print(neighbors)
#print(test_accuracies)

#Plot it!
plt.scatter(neighbors, test_accuracies)
plt.xlabel("Number of Neighbors")
plt.ylabel("Test Accuracy")
plt.title("Number of Neighbors vs. Test Accuracy")

plt.show()

**Results**

Test accuracy generally trended upward as k increased, though not monotonically. Accuracy for k < 3 was noticeably lower than for k = 3 and up, and even k = 3–4 lagged behind k = 5+, which lines up with 5 being a common default minimum for k in practice. Accuracy peaked around k = 10, dipped slightly, then rose again toward k = 15 — a less clean trend than a simple "higher k is always better" pattern, and worth investigating further with a different dataset or wider k range.

## KNN Regression (Housing Price Dataset)

### An Ethically Inappropriate Feature

This dataset includes a feature described as *"1000(Bk − 0.63)² where Bk is the proportion of Black residents by town."*

This is an ethically inappropriate input feature. While race can correlate with many social and economic outcomes due to historical inequities, encoding it directly as a numeric predictor of housing price — and singling out this one demographic feature while excluding any others — implies that racial composition should causally influence a home's value. Including it this way risks reinforcing exactly the kind of historical, discriminatory pricing patterns that fair housing practices are meant to eliminate. The appropriate choice is to exclude this feature entirely from the model.

### Regression Performance Across Preprocessing Choices

KNN regression (k=3) on the housing dataset, comparing no preprocessing, normalization only, and normalization with distance weighting — reporting MAE and coefficient of determination (R²) for train and test sets in each case.

In [ ]:
# Learn housing prices data
# Create Data Frames from the data files
train_data, train_meta = arff.loadarff("data/housing-prices/housing_train.arff")

prices_df= pd.DataFrame(train_data)

#EDA
print("First 5 rows of housing prices data set:")
display(prices_df.head())

print("\nHousing Price (MEDV) Distribution:")
display(prices_df["MEDV"].value_counts())

print(f"Data shape: {prices_df.shape}")

#split into x and y
X = prices_df.drop(["MEDV"], axis = 1)
y = prices_df["MEDV"]

#Function for running each split
def knn_regression(X, y, normalization = False, weights="uniform", test_size = 0.2, k = 3, random_state = np.random.randint(0, 1000)):
    """Initialize and fit a KNNeighbors Regression Model that can have normalization and/or
    weighting on hyperparameters.
    Calculates and returns the train and test MAE and coefficient of determination.
    """
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size = test_size, random_state = random_state #randomly generated state // in the func so i can set it in the next part
    )

    knn_model = KNeighborsRegressor(n_neighbors = k, weights = weights)

    if normalization:
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        knn_model.fit(X_train_scaled, y_train)
        y_train_pred = knn_model.predict(X_train_scaled)
        y_test_pred = knn_model.predict(X_test_scaled)
    else:
        knn_model.fit(X_train, y_train)
        y_train_pred = knn_model.predict(X_train)
        y_test_pred = knn_model.predict(X_test)

    train_mae = mean_absolute_error(y_train, y_train_pred)
    train_r2 = r2_score(y_train, y_train_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)
    test_r2 = r2_score(y_test, y_test_pred)

    return train_mae, train_r2, test_mae, test_r2

print("\nRunning KNNeighbors Regression with k = 3, NO weighting, and NO normalization:")
train_mae, train_r2, test_mae, test_r2 = knn_regression(X, y)
print(f"\n\tTrain MAE: {train_mae}\n\t Train Coefficient of Determination: {train_r2}")
print(f"\n\tTest MAE: {test_mae}\n\t Test Coef. of Determination: {test_r2}")

print("\nRunning KNNeighbors Regression with k = 3, NO weighting, and normalization:")
train_mae, train_r2, test_mae, test_r2 = knn_regression(X, y, True)
print(f"\n\tTrain MAE: {train_mae}\n\t Train Coefficient of Determination: {train_r2}")
print(f"\n\tTest MAE: {test_mae}\n\t Test Coef. of Determination: {test_r2}")

print("\nRunning KNNeighbors Regression with k = 3, weighting, and normalization:")
train_mae, train_r2, test_mae, test_r2 = knn_regression(X, y, True, "distance")
print(f"\n\tTrain MAE: {train_mae}\n\t Train Coefficient of Determination: {train_r2}")
print(f"\n\tTest MAE: {test_mae}\n\t Test Coef. of Determination: {test_r2}")


**Results**

*(Note: no random seed was fixed for these runs, so exact values will vary slightly between executions — the pattern below held consistently across runs.)*

| Configuration | Train MAE | Train R² | Test MAE | Test R² |
|---|---|---|---|---|
| No normalization, no weighting | ~2.9 | ~0.76 | ~4.16 | ~0.62 |
| Normalization only | ~1.8 | ~0.77 | ~3.1 | ~0.89 |
| Normalization + distance weighting | 0.0 | 1.0 | ~2.88 | ~0.81 |

The unweighted, unnormalized model showed a large gap between train and test MAE (2.9 vs. 4.16) — a sign of overfitting to the training data, unsurprising given no hyperparameters were tuned to the dataset's scale. Adding normalization closed that gap substantially (1.8 vs. 3.1) and improved both R² values. Adding distance weighting on top pushed train error to 0 (perfect fit to training data, R² = 1.0) while further lowering test MAE to the best result of the three configurations. As expected, both normalization and distance weighting reduce model bias — each preprocessing step lowered MAE and raised R² on the test set, with the combination of both performing best overall.

### Varying k for Regression

In [ ]:
# Learn and graph for different k values
neighbors = [k for k in range(1, 16)]
test_maes = []

for k in neighbors:
    _, _, test_mae, _ = knn_regression(X, y, True, "distance", 0.2, k, 10)
    test_maes.append(test_mae)

plt.clf()
plt.scatter(neighbors, test_maes)
plt.xlabel("Number of Neighbors")
plt.ylabel("Test MAE")
plt.title("Number of Neighbors vs. Test MAE")

plt.show()

**Results**

Test MAE increased as k increased — the same directional trend seen with test accuracy in the classification section (part 2), where increasing k tends to make the model smoother but less locally precise. With more neighbors contributing to each prediction, points that should carry less influence get included anyway, smoothing the decision/regression boundary — potentially past the point that's useful, introducing bias that pulls predictions in a consistently wrong direction.

To confirm distance weighting and normalization were meaningfully helping, the same experiment was re-run without them: the entire MAE curve shifted up by more than 2 points. So while the errors seen here (~3+) were still somewhat higher than initially expected, normalization and distance weighting cut the error nearly in half compared to the unprocessed version — a clear demonstration of why these preprocessing steps matter in practice.

## A Custom Distance Metric for Mixed Nominal/Continuous Data

The Lymphography dataset contains both continuous and nominal attributes. This section implements a custom distance function that uses Euclidean distance for continuous features and 0/1 (Hamming) distance for nominal features, then compares it against a model that treats all features as continuous.

In [ ]:
# Train/Predict lymph with your own distance metric

# Lymph attributes list
lymph_attributes = ["class",
                    "lymphatics",
                    "block of affere",
                    "bl. of lymph. c",
                    "bl. of lymph. s",
                    "by pass",
                    "extravasates",
                    "regeneration of",
                    "early uptake in",
                    "lym.nodes dimin",
                    "lym.nodes enlar",
                    "changes in lym.",
                    "defect in node",
                    "changes in node",
                    "changes in stru",
                    "special forms",
                    "dislocation of",
                    "exclusion of no",
                    "no. of nodes in"
                    ]

# Read CSV, create df
lymph_df = pd.read_csv("data/lymphography/lymphography.data", header = None, names = lymph_attributes)

# EDA
print("EDA:")
print("\tFirst 5 rows of lymph data set:")
display(lymph_df.head())

print("\n\tLymph Class Distribution:")

print(f"\tData shape: {lymph_df.shape}")

# Split into X and y
X = lymph_df.drop(["class"], axis = 1)
y = lymph_df["class"]

# Convert Nominal features
nominal_attributes = ["lymphatics", 
                      "block of affere",
                      "bl. of lymph. c",
                      "bl. of lymph. s",
                      "by pass",
                      "extravasates",
                      "regeneration of",
                      "early uptake in",
                      "changes in node",
                      "changes in stru",
                      "special forms",
                      "dislocation of",
                      "exclusion of no",
                      ]

for attr in nominal_attributes:
    le = LabelEncoder()
    X[attr] = le.fit_transform(X[attr])

nominal_indices = [X.columns.get_loc(attr) for attr in nominal_attributes]

# Distance function(s)
def mydist(x1, x2):
    """Given two arrays (x1, x2), compute 0/1 distance or Euclidean distance
    depending on nominal or continuous attributes. 
    Returns the distance as a float."""
    dist_nominal = 0
    dist_continuous = 0
    for i in range(len(x1)):
        if i in nominal_indices:
            # implement 0/1 distance
            if x1[i] != x2[i]:
                dist_nominal += 1
        else:
            #implement euclidean distance
            dist_continuous += (x1[i] - x2[i])**2
    return dist_nominal + np.sqrt(dist_continuous)

# Split Data
X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size = 0.2, stratify=y)

# Initialize/Fit KNeighbors Classifiera
knn = KNeighborsClassifier(n_neighbors = 5, metric = mydist)
knn.fit(X_train, y_train)

knn_all_continuous = KNeighborsClassifier(n_neighbors = 5)
knn_all_continuous.fit(X_train, y_train)
#Check Accuracies 
train_acc_1 = knn.score(X_train, y_train)
test_acc_1 = knn.score(X_test, y_test)

train_acc_2 = knn_all_continuous.score(X_train, y_train)
test_acc_2 = knn_all_continuous.score(X_test, y_test)

print("\nTrain and Test Accuracies (when using a different distance metric for continuous and nominal values):")
print(f"\n\tTrain Accuracy: {train_acc_1:.4f}")
print(f"\tTest Accuracy: {test_acc_1:.4f}")

print("\nTrain and Test Accuracies (when treating all values as continuous):")
print(f"\n\tTrain Accuracy: {train_acc_2:.4f}")
print(f"\tTest Accuracy: {test_acc_2:.4f}")

**Approach**

The custom `mydist` function tracks which feature indices are nominal. For each pair of points, it applies 0/1 distance to nominal features (incrementing a counter on any mismatch) and Euclidean distance to continuous features, then combines them as `nominal_mismatches + sqrt(sum of squared continuous differences)`. This custom-metric model is compared against a standard `KNeighborsClassifier` using default Euclidean distance across all features.

**Results**

The custom-metric model's train and test accuracy varied noticeably depending on random state — sometimes train was higher, sometimes test was — suggesting the model is quite sensitive to how the data happens to be split, which makes sense given it fits more precisely to the specific structure of mixed-type data.

The all-continuous model was far more stable across random states, with train and test accuracy staying close together consistently. Treating nominal features as continuous appears to produce a more uniform, less split-sensitive model — but likely at the cost of not capturing the true structure of categorical attributes as well as the mixed-metric approach does. In practice, the right choice depends on the situation: a custom mixed-type metric is more expressive but more sensitive to how the data is split, while treating everything as continuous is more stable but a coarser approximation.

## Conclusion

Across both classification and regression tasks, the same core pattern held: normalization and distance weighting consistently improved model performance, and smaller values of k tended to generalize better than large ones, since KNN's predictions grow smoother (and more biased) as more neighbors are averaged together. The custom distance metric for the Lymphography dataset also highlighted a real tradeoff in KNN — a metric tailored to the data's actual structure (mixed nominal/continuous) can capture more nuance, but at the cost of stability across different data splits.

A natural extension would be implementing KNN from scratch (classification and regression, with optional distance weighting) to compare its behavior directly against scikit-learn's implementation.